In [1]:

import os
import sys
import argparse
import datetime
import numpy as np
import time
import torch
import torch.backends.cudnn as cudnn
import json
from pathlib import Path
from collections import OrderedDict
from functools import partial

# Add project root to path
# sys.path.insert(0, str(Path(__file__).parent))

from timm.models import create_model
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from src.optim.optim_factory import create_optimizer, get_parameter_groups, LayerDecayValueAssigner

from timm.utils import ModelEma

from src.utils.config import get_cfg, merge_config_file, freeze_cfg, load_and_freeze_config
from src.engine.train_engine import TrainingEngine
from src.engine.val_engine import ValidationEngine
from src.utils.evaluation import merge_distributed_results
from src.optim.mixup import Mixup
# from src.optim.optim_factory import LayerDecayValueAssigner
from src.dataset.datasets import build_dataset
from src.utils.utils import NativeScalerWithGradNormCount as NativeScaler
from src.utils.utils import multiple_samples_collate
from src.utils.logger import TensorboardLogger
from src.utils import utils

from src.models import ViT, ViT_pretrain, layers

from run_finetuning_with_yacs import create_data_loaders, create_model_from_config

/root/miniconda3/envs/mamba/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/mamba/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/root/miniconda3/envs/mamba/lib/python3.12/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [2]:
all_data = []

csv_path = "/home/qzk/Facial-Foundation-Model/saved/data/dfew_224/org/split01/test.csv"
all_folders = []

import pandas as pd
import cv2
df = pd.read_csv(csv_path)
folder_lbls = list(df.values[:,0])
all_folders = [folder_lbl.split(" ")[0] for folder_lbl in folder_lbls]
all_lbls = [folder_lbl.split(" ")[1:] for folder_lbl in folder_lbls]
print("we have ", len(all_folders), " folders in total, like", all_folders[0])
print("we have ", len(all_lbls), " lbls in total, like", all_lbls[0])


for i, folder in enumerate(all_folders):
    if not os.path.isdir(folder):
        continue
    imgs = []
    all_imgs_path = os.listdir(folder)
    for img_path in all_imgs_path:
        img = cv2.imread(os.path.join(folder, img_path))
        if img is not None:
            imgs.append(img)
    all_data.append({"folder": all_folders[i], "images": imgs, "label": all_lbls[i]})
    
print("len of all_data", len(all_data))


we have  2340  folders in total, like /root/shared/emotion_dataset/DFEW/clip_244x244_16f/00002
we have  2340  lbls in total, like ['2']
len of all_data 2340


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class relblDataset(Dataset):
    def __init__(self, all_data):
        self.all_data = all_data

    def __len__(self):
        return len(self.all_data)

    def __getitem__(self, idx):
        item = self.all_data[idx]
        images = item["images"]
        label = item["label"]
        folder = item["folder"]
        # if self.transform:
        #     images = [self.transform(image=image)["image"] for image in images]
        img_tensor = torch.tensor(images)/255.0
        img_tensor = img_tensor.unsqueeze(0)
        # data_transform = video_transforms.Compose([
        #     # video_transforms.Resize(size=(160, 160), interpolation='bilinear'),
        #     # volume_transforms.ClipToTensor(),
        #     video_transforms.Normalize(mean=[0.485, 0.456, 0.406],
        #                                 std=[0.229, 0.224, 0.225])
        # ])
        # img_tensor = data_transform(img_tensor)
        # print("shape of img_tensor", img_tensor.shape) # ([1, 16, 224, 224, 3])
        # interpolate to 160*160, but use nn.interpolate
        img_tensor = F.interpolate(img_tensor, size = (160, 160, 3))
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 1, 1, 3)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 1, 1, 3)
        img_tensor = (img_tensor - mean) / std
        
        img_tensor = img_tensor.permute(0, 4, 1, 2, 3)  # Change to [B, C, T, H, W]
        img_tensor = img_tensor.squeeze(0)
        return {"images": img_tensor, "label": label, "folder": folder}

data_loader = DataLoader(relblDataset(all_data), batch_size=32, shuffle=False, num_workers=20)

In [11]:
sample = next(iter(data_loader) )
print("sample images:", sample["images"].shape)

sample images: torch.Size([32, 3, 16, 160, 160])


In [ ]:
# visualize some data
import matplotlib.pyplot as plt
print("Visualizing some data")
for j in range(10):
    eg_data = all_data[j]
    print("eg_data", eg_data["folder"], "has", len(eg_data["images"]), "images, with label: ", eg_data["label"])
    # show images
    plt.figure(figsize=(20, 20))
    for i in range(len(eg_data["images"])):
        plt.subplot(4, 4, i + 1)
        plt.imshow(eg_data["images"][i])
        plt.axis('off')
    plt.show()


In [11]:
config_path = 'configs/gazeCapture.yaml'
cfg = get_cfg()
merge_config_file(cfg, config_path)
model = create_model_from_config()
model_path = "/home/qzk/Facial-Foundation-Model/output/gazeCapture_8_11/checkpoint-best.pth"
state_dict = torch.load(model_path, map_location='cpu', weights_only=False)
state_dict = state_dict["model"]
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

==> Note: Use 'local_global' for compute reduction (lg_region_size=[2, 2, 10],lg_first_attn_type=self, lg_third_attn_type=cross,lg_attn_param_sharing_first_third=False,lg_attn_param_sharing_all=False,lg_classify_token_type=org,lg_no_second=False, lg_no_third=False)
==> Number of local regions: 20 (size=[4, 5, 1])
model after create_model = VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv3d(3, 512, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): LGBlock(
      (first_attn_norm0): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (first_attn): GeneralAttention(
        (q): Linear(in_features=512, out_features=512, bias=False)
        (kv): Linear(in_features=512, out_features=1024, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (second_a

<All keys matched successfully>

In [12]:
import src.dataset.augment.video_transforms as video_transforms 
import src.dataset.augment.volume_transforms as volume_transforms
import torch.nn.functional as F
model.eval()
for i, data in enumerate(all_data):
    img_tensor = torch.tensor(data["images"])/255.0
    img_tensor = img_tensor.unsqueeze(0)
    # data_transform = video_transforms.Compose([
    #     # video_transforms.Resize(size=(160, 160), interpolation='bilinear'),
    #     # volume_transforms.ClipToTensor(),
    #     video_transforms.Normalize(mean=[0.485, 0.456, 0.406],
    #                                 std=[0.229, 0.224, 0.225])
    # ])
    # img_tensor = data_transform(img_tensor)
    # print("shape of img_tensor", img_tensor.shape) # ([1, 16, 224, 224, 3])
    # interpolate to 160*160, but use nn.interpolate
    img_tensor = F.interpolate(img_tensor, size = (160, 160, 3))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 1, 1, 3)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 1, 1, 3)
    img_tensor = (img_tensor - mean) / std
    
    img_tensor = img_tensor.permute(0, 4, 1, 2, 3)  # Change to [B, C, T, H, W]

    with torch.no_grad():
        output = model(img_tensor)
        data["gaze"] = output.detach().cpu().numpy()
    
    if i > 20:
        break
    
    
print("shape of output", all_data[0]["gaze"].shape)

shape of output (1, 32)


In [ ]:
# # visualize some data
# import matplotlib.pyplot as plt
# print("Visualizing some data")
# for j in range(10):
#     eg_data = all_data[j]
#     print("eg_data", eg_data["folder"], "has", len(eg_data["images"]), "images, with label: ", eg_data["label"])
#     # show images
#     img_tensor = torch.tensor(eg_data["images"])
#     img_tensor = img_tensor.unsqueeze(0)
#     # data_transform = video_transforms.Compose([
#     #     # video_transforms.Resize(size=(160, 160), interpolation='bilinear'),
#     #     # volume_transforms.ClipToTensor(),
#     #     video_transforms.Normalize(mean=[0.485, 0.456, 0.406],
#     #                                 std=[0.229, 0.224, 0.225])
#     # ])
#     # img_tensor = data_transform(img_tensor)
#     # print("shape of img_tensor", img_tensor.shape) # ([1, 16, 224, 224, 3])
#     # interpolate to 160*160, but use nn.interpolate
#     img_tensor = F.interpolate(img_tensor, size = (160, 160, 3))
#     mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 1, 1, 3)
#     std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 1, 1, 3)
#     img_tensor = (img_tensor/255.0 - mean) / std
    
#     img_tensor = img_tensor.permute(0, 4, 1, 2, 3)  # Change to [B, C, T, H, W]
#     sample = img_tensor.squeeze(0)
#     # record to a video file
#     sample = sample.permute(1, 2, 3, 0)  # Change to (time, height, width, channels)
#     sample = sample.detach().cpu().numpy()  # Convert to numpy array
#     # add mean and std normalization
#     mean = np.array([0.485, 0.456, 0.406])
#     std = np.array([0.229, 0.224, 0.225])
#     sample = (sample * std + mean)  # Denormalize
#     sample = np.clip(sample, 0, 1)  # Clip values to [0, 1]
#     sample = (sample * 255).astype(np.uint8)  # Convert to uint8
    
    
#     plt.figure(figsize=(20, 20))
#     for i in range(len(eg_data["images"])):
#         plt.subplot(4, 4, i + 1)
#         plt.imshow(sample[i])
#         plt.axis('off')
#     plt.show()


In [14]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# if target_angle.shape[1] == 3:
#     from src.utils.gaze import gaze3d_to_gaze2d
#     target_angle = gaze3d_to_gaze2d(target_angle)

for i, data in enumerate(all_data):
    img_tensor = torch.tensor(data["images"])/255.0
    img_tensor = img_tensor.unsqueeze(0)
    # data_transform = video_transforms.Compose([
    #     # video_transforms.Resize(size=(160, 160), interpolation='bilinear'),
    #     # volume_transforms.ClipToTensor(),
    #     video_transforms.Normalize(mean=[0.485, 0.456, 0.406],
    #                                 std=[0.229, 0.224, 0.225])
    # ])
    # img_tensor = data_transform(img_tensor)
    # print("shape of img_tensor", img_tensor.shape) # ([1, 16, 224, 224, 3])
    # interpolate to 160*160, but use nn.interpolate
    img_tensor = F.interpolate(img_tensor, size = (160, 160, 3))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 1, 1, 3)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 1, 1, 3)
    img_tensor = (img_tensor - mean) / std
    
    img_tensor = img_tensor.permute(0, 4, 1, 2, 3)
    sample = img_tensor.squeeze(0)
    # record to a video file
    sample = sample.permute(1, 2, 3, 0)  # Change to (time, height, width, channels)
    sample = sample.detach().cpu().numpy()  # Convert to numpy array
    # add mean and std normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    sample = (sample * std + mean)  # Denormalize
    sample = np.clip(sample, 0, 1)  # Clip values to [0, 1]
    sample = (sample * 255).astype(np.uint8)  # Convert to uint8
    pred = data["gaze"].reshape(-1, 2)
    new_samples = []
    for j in range(sample.shape[0]):
        # Draw the target and output angles on each frame
        output_angle = pred[j]
        
        pl0 = -np.cos(output_angle[0]) * np.sin(output_angle[1])
        pl1 = -np.sin(output_angle[0])
        
        new_sample = cv2.arrowedLine(sample[j].copy(), (80, 80), (int(80 + pl0 * 50), int(80 + pl1 * 50)), (0, 0, 255), 2, tipLength=0.1)
        
        new_samples.append(new_sample)
        
    new_samples = np.array(new_samples)
    sample = new_samples  # Use the modified samples with arrows
        # cv2.arrowedLine(sample[j], (80, 80), (int(80 + ol0 * 50), int(80 + ol1 * 50)), (255, 0, 0), 2, tipLength=0.1)
    # sample = cv2.cvtColor(sample, cv2.COLOR_RGB2BGR)
    # record a video file
    video_path = f"output/data_relbl/sample_video_{i}.mp4"
    
    # if not os.path.exite()
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    height, width, _ = sample[0].shape
    out = cv2.VideoWriter(video_path, fourcc, 5.0, (width, height))
    for frame in sample:
        out.write(frame)
    out.release()
    print("video saved to:", video_path)
    
    if i > 20:
        break

video saved to: output/data_relbl/sample_video_0.mp4
video saved to: output/data_relbl/sample_video_1.mp4
video saved to: output/data_relbl/sample_video_2.mp4
video saved to: output/data_relbl/sample_video_3.mp4


video saved to: output/data_relbl/sample_video_4.mp4
video saved to: output/data_relbl/sample_video_5.mp4
video saved to: output/data_relbl/sample_video_6.mp4
video saved to: output/data_relbl/sample_video_7.mp4
video saved to: output/data_relbl/sample_video_8.mp4
video saved to: output/data_relbl/sample_video_9.mp4
video saved to: output/data_relbl/sample_video_10.mp4
video saved to: output/data_relbl/sample_video_11.mp4
video saved to: output/data_relbl/sample_video_12.mp4
video saved to: output/data_relbl/sample_video_13.mp4
video saved to: output/data_relbl/sample_video_14.mp4
video saved to: output/data_relbl/sample_video_15.mp4
video saved to: output/data_relbl/sample_video_16.mp4
video saved to: output/data_relbl/sample_video_17.mp4
video saved to: output/data_relbl/sample_video_18.mp4
video saved to: output/data_relbl/sample_video_19.mp4
video saved to: output/data_relbl/sample_video_20.mp4
video saved to: output/data_relbl/sample_video_21.mp4
